# PySyft x Sovereign Mohawk PoC

This notebook demonstrates client-side FL aggregation where PySyft controls remote execution policies and Mohawk handles robust aggregation and proof verification.

In [ ]:
import numpy as np
from collections import defaultdict

try:
    from mohawk import MohawkNode
except Exception:
    MohawkNode = None

In [ ]:
def mock_local_train(model_params=None, seed=0):
    rng = np.random.default_rng(seed)
    if model_params is None:
        model_params = {
            'layer1.weight': rng.normal(0.0, 0.1, size=(4, 4)).astype(np.float32),
            'layer1.bias': rng.normal(0.0, 0.1, size=(4,)).astype(np.float32),
        }
    updated = {k: v + rng.normal(0.0, 0.01, size=v.shape).astype(np.float32) for k, v in model_params.items()}
    metrics = {'accuracy': float(np.clip(0.80 + rng.normal(0.0, 0.02), 0.0, 1.0)), 'loss': float(np.clip(0.35 + rng.normal(0.0, 0.03), 0.0, 10.0))}
    return metrics, updated

def average_aggregate(updates):
    keys = list(updates[0].keys())
    return {k: np.mean(np.stack([u[k] for u in updates], axis=0), axis=0) for k in keys}

In [ ]:
def mohawk_aggregate(updates, node=None):
    if node is None:
        return average_aggregate(updates), {'engine': 'numpy-average', 'proof_verified': False}

    keys = list(updates[0].keys())
    flattened = [np.concatenate([np.ravel(u[k]).astype(np.float32) for k in keys]) for u in updates]
    payload = [{'node_id': f'ds_{i}', 'gradient': vec.tolist()} for i, vec in enumerate(flattened)]
    result = node.aggregate(payload)

    proof_verified = False
    if isinstance(result, dict) and result.get('proof') is not None and hasattr(node, 'verify_proof'):
        proof_verified = bool(node.verify_proof(result['proof']))

    vector = None
    if isinstance(result, dict):
        for key in ('aggregated', 'gradient', 'aggregated_gradient', 'result'):
            if result.get(key) is not None:
                vector = np.asarray(result[key], dtype=np.float32)
                break

    if vector is None:
        vector = np.mean(np.stack(flattened, axis=0), axis=0)

    restored = {}
    offset = 0
    for key in keys:
        shape = updates[0][key].shape
        size = int(np.prod(shape))
        restored[key] = vector[offset:offset + size].reshape(shape)
        offset += size

    return restored, {'engine': 'mohawk', 'proof_verified': proof_verified}

In [ ]:
rounds = 5
participants = 5
metrics = defaultdict(list)
model_params = None

node = None
if MohawkNode is not None:
    try:
        node = MohawkNode()
        if hasattr(node, 'start'):
            node.start(config_path='capabilities.json')
    except Exception:
        node = None

for epoch in range(rounds):
    epoch_updates = []
    for pid in range(participants):
        m, p = mock_local_train(model_params, seed=epoch * 100 + pid)
        metrics[epoch].append(m)
        epoch_updates.append(p)

    model_params, info = mohawk_aggregate(epoch_updates, node=node)
    mean_acc = float(np.mean([x['accuracy'] for x in metrics[epoch]]))
    mean_loss = float(np.mean([x['loss'] for x in metrics[epoch]]))
    print(f'epoch={epoch} engine={info[
]} proof_verified={info[
]} accuracy={mean_acc:.4f} loss={mean_loss:.4f}')

## Datasite Mode Snippet

Use the script for live Datasite execution:

`python examples/pysyft-integration/pysyft_mohawk_poc.py --mode datasite --server-url http://localhost:8080 --email you@example.com --password ... --dataset YourDataset --asset YourData`